# Hafta 1 - Geliştirilmiş analiz (v2)
Bu defter 02_hafta01_analiz_v2'den sonra çalıştırılır. Girdi: `02_islenmis/ana_tablo_saatlik.csv`.

Bölümler: 1) USD bazlı aynı-ay karşılaştırması, 2) regresyon, 3) arz eğrisi kayması (net yük), 4) barajlı/akarsu gün içi profili, 5) yağış (Open-Meteo, internet gerekir).

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, statsmodels.formula.api as smf
CSV="02_islenmis/ana_tablo_saatlik.csv"
df=pd.read_csv(CSV,parse_dates=["zaman"]).set_index("zaman")
df["y"]=df.index.year; df["m"]=df.index.month; df["h"]=df.index.hour
df["hidro"]=(df.uretim_dammedHydro+df.uretim_river)/1000
df["net"]=(df.tuketim_mwh-df.uretim_wind-df.uretim_solar)/1000
print(df.shape)

## 1) USD bazlı, aynı-ay karşılaştırması
2026'da Kasım-Aralık yok. Yıl ortalamasına oranlamak 2026 yaz-sonbaharını yukarı yanlı gösterir; bu yüzden 2026'yı geçmiş yılların *aynı ay* ortalamasıyla kıyaslıyoruz.

In [ ]:
u=df.groupby(["y","m"]).ptf_usd.mean().unstack(0)
oran=(u[2026]/u[[2023,2024,2025]].mean(axis=1)).dropna()
display((oran*100).round(0).rename("2026 / 2023-25 aynı ay (%)"))
oran.mul(100).plot.bar(figsize=(8,4),color="#9aa1ad"); plt.axhline(100,ls="--",c="k"); plt.show()

## 2) Regresyon
Saatlik USD PTF ~ hidro + rüzgâr + güneş + tüketim + saat/ay/yıl sabit etkileri, Newey-West (24 gecikme). Katsayı: 1 GW artışın USD/MWh etkisi. Doğal gaz/kömür fiyatı ve fiyat tavanı modelde YOK.

In [ ]:
d=df.dropna(subset=["ptf_usd"]).copy()
d["wp"]=d.uretim_wind/1000; d["sp"]=d.uretim_solar/1000; d["tk"]=d.tuketim_mwh/1000
r=smf.ols("ptf_usd ~ hidro+wp+sp+tk+C(h)+C(m)+C(y)",d).fit(cov_type="HAC",cov_kwds={"maxlags":24})
print(r.params[["hidro","wp","sp","tk"]].round(2)); print("R2",round(r.rsquared,3))
for dr in ["hidro","wp","sp","tk"]:
    t=[v for v in ["hidro","wp","sp","tk"] if v!=dr]
    print("R2 (",dr,"çıkarılınca):",round(smf.ols("ptf_usd ~ "+"+".join(t)+"+C(h)+C(m)+C(y)",d).fit().rsquared,3))
s=d[(d.y==2026)&d.m.isin([4,5,6])].copy(); h25=d[(d.y==2025)&d.m.isin([4,5,6])].hidro.mean()
s2=s.copy(); s2["hidro"]=s.hidro-(s.hidro.mean()-h25)
print("Nis-Haz 2026 gerçek:",round(s.ptf_usd.mean(),1),"| model:",round(r.predict(s).mean(),1),"| hidro 2025 seviyesinde:",round(r.predict(s2).mean(),1))

## 3) Arz eğrisi kayması
Aynı net yük aralığında 2025 ve 2026 Nisan-Haziran fiyatları.

In [ ]:
s=df[df.m.isin([4,5,6])&df.y.isin([2025,2026])].copy()
s["b"]=pd.cut(s.net,[0,20,25,30,35,40,60],labels=["<20","20-25","25-30","30-35","35-40",">40"])
t=s.groupby(["b","y"],observed=True).ptf_usd.mean().unstack(); display(t.round(1))
t.plot.bar(figsize=(8,4),color=["#9aa1ad","#d9480f"]); plt.ylabel("USD/MWh"); plt.show()

## 4) Barajlı ve akarsu hidro, gün içi profil

In [ ]:
p=s.groupby(["y","h"])[["uretim_dammedHydro","uretim_river"]].mean()/1000
fig,ax=plt.subplots(figsize=(8,4))
for y,c in [(2025,"#9aa1ad"),(2026,"#d9480f")]:
    ax.plot(p.loc[y].index,p.loc[y].uretim_dammedHydro,c=c,lw=3,label=f"Barajlı {y}")
    ax.plot(p.loc[y].index,p.loc[y].uretim_river,c=c,ls="--",label=f"Akarsu {y}")
ax.legend(); ax.set_ylabel("GW"); plt.show()

## 5) Yağış (Open-Meteo)
Ücretsiz, ticari olmayan kullanım için. Büyük hidro havzalarından birkaç nokta seçip günlük toplam yağışı çeker, aylığa toplar; hidro üretimle 0-2 ay gecikmeli ilişkisine bakar (kar erimesi gecikme yaratabilir). Nokta listesi yaklaşıktır, havza ağırlıkları temsili değildir.

In [ ]:
import requests, time
NOKTALAR={"Elazığ (Fırat)":(38.7,39.2),"Şanlıurfa (Atatürk Barajı)":(37.5,38.4),"Artvin (Çoruh)":(41.2,41.8),
          "Sivas (Kızılırmak)":(39.7,37.0),"Erzurum (Doğu Anadolu)":(39.9,41.3),"Diyarbakır (Dicle)":(37.9,40.2)}
def yagis(lat,lon):
    r=requests.get("https://archive-api.open-meteo.com/v1/archive",params=dict(latitude=lat,longitude=lon,
        start_date="2023-01-01",end_date="2026-10-31",daily="precipitation_sum",timezone="Europe/Istanbul"),timeout=60)
    r.raise_for_status(); j=r.json()["daily"]
    return pd.Series(j["precipitation_sum"],index=pd.to_datetime(j["time"]))
try:
    Y=pd.DataFrame({k:yagis(*v) for k,v in NOKTALAR.items()}); time.sleep(1)
    ay=Y.mean(axis=1).resample("MS").sum().rename("yagis_mm")
    hid=df.hidro.resample("MS").mean().rename("hidro_gw")
    M=pd.concat([ay,hid],axis=1).dropna()
    for lag in (0,1,2):
        print(f"gecikme {lag} ay: korelasyon =",round(M.yagis_mm.shift(lag).corr(M.hidro_gw),2))
    M.to_csv("02_islenmis/yagis_hidro_aylik.csv"); display(M.tail(12))
except Exception as e:
    print("Yağış verisi alınamadı:",e)

Çıktıyı (özellikle korelasyon satırlarını ve `yagis_hidro_aylik.csv` dosyasını) bana gönderirsen carousel'e yağış sayfasını eklerim. Korelasyon nedensellik değildir; kar erimesi ve baraj rezervi ilişkiyi geciktirir.

## 6) Yıllara göre gün içi profil ve güneş/rüzgâr payı
Her yılın Oca-Eki dönemi (2026'da Kasım-Aralık yok). Güneş: EPİAŞ gerçek zamanlı üretimdeki 'sun' alanı; lisanssız üretimi tam kapsamayabilir.

In [ ]:
d=df[df.m<=10].copy()
d["gun"]=np.where(d.h.between(10,15),"ogle",np.where(d.h.between(17,21),"aksam","diger"))
t=d.groupby(["y","gun"]).ptf_usd.mean().unstack(); t["ort"]=d.groupby("y").ptf_usd.mean()
t["ogle/ort"]=t.ogle/t.ort; t["aksam/ogle"]=t.aksam/t.ogle; display(t.round(2))
p=d.groupby(["y","h"]).ptf_usd.mean().unstack(0); p.plot(figsize=(8,4)); plt.ylabel("USD/MWh"); plt.show()
d["gunes_pay"]=d.uretim_solar/d.uretim_total*100; d["ruzgar_pay"]=d.uretim_wind/d.uretim_total*100
ogle=d[d.h.between(10,15)]
display(ogle.groupby("y")[["gunes_pay","ruzgar_pay"]].mean().round(1))
o=ogle.groupby(["y","m"]).agg(g=("gunes_pay","mean"),r=("ruzgar_pay","mean"),f=("ptf_usd","mean")); o["yenil"]=o.g+o.r
print("korelasyon (öğle yenilenebilir payı, öğle fiyatı):",round(o.yenil.corr(o.f),2))
plt.scatter(o.yenil,o.f); plt.xlabel("Öğle güneş+rüzgâr payı (%)"); plt.ylabel("Öğle PTF (USD/MWh)"); plt.show()